## Env variable

In [21]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("DEEPSEEK_API_KEY")

## Create Agents

### Time keeper agent

In [22]:
import datetime
from langchain_deepseek import ChatDeepSeek
from langgraph.prebuilt import create_react_agent

ds = ChatDeepSeek(
    model="deepseek-reasoner", # reasoning_content field issue
    temperature=0,
    max_tokens=None,
    max_retries=2,
)

def current_datetime() -> str:
    "This tool provides the current date, time and day of the week in your location."
    dt = datetime.datetime.now()
    return "Date time is {} and today is {}".format(dt, dt.strftime('%A'))

time_keeper_agent = create_react_agent(
    model=ds,
    tools=[current_datetime],
    name="time_keeper_agent",
    prompt="You tell the current date, time and day of the week in your location."
)

/var/folders/bq/46md1pc9595b4b_x2_hfjg9w0000gn/T/ipykernel_6234/2080528221.py:17: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  time_keeper_agent = create_react_agent(


### Schedule Keeper Agent 

In [23]:
def shop_open_hours() -> str:
    "This tool provides the daily opening hours schedule for a shop."
    schedule = """Monday to Friday: 10:00am to 08:00pm
                Saturday: 9:00am to 08:00pm
                Sunday: 1:00pm to 06:00pm"""
    return schedule

schedule_keeper_agent = create_react_agent(
    model=ds,
    tools=[shop_open_hours],
    name="schedule_keeper_agent",
    prompt="You tell the shop's operating hours for each day in a week."
)

/var/folders/bq/46md1pc9595b4b_x2_hfjg9w0000gn/T/ipykernel_6234/495407217.py:8: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  schedule_keeper_agent = create_react_agent(


## Create Supervisor

In [24]:
from langgraph_supervisor import create_supervisor

prompt = ("""You are a team supervisor to manage time_keeper_agent and schedule_keeper_agent
Your job is to respond to shop opening hour queries using the two agents. To get the current time and date, use time_keeper_agent;
To get shop's operating hours, use schedule_keeper_agent.
You can use both agents in parallel to get required information.
""")

supervisor = create_supervisor(
    [time_keeper_agent, schedule_keeper_agent],
    model=ds,
    output_mode="full_history",
    prompt=prompt
)

In [25]:
app = supervisor.compile()
result = app.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What are the shop's opening hours on the weekend? Is the shop open right now?"
        }
    ]
})

for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

What are the shop's opening hours on the weekend? Is the shop open right now?
================================== Ai Message ==================================
Name: supervisor

I'll find out both the shop's weekend hours and whether it's open right now. Let me check with both agents in parallel!
Tool Calls:
  transfer_to_time_keeper_agent (call_00_rbkz5ttjWODTg0htH5sW2142)
 Call ID: call_00_rbkz5ttjWODTg0htH5sW2142
  Args:
================================= Tool Message =================================
Name: transfer_to_time_keeper_agent

Successfully transferred to time_keeper_agent
================================== Ai Message ==================================
Name: time_keeper_agent

Let me start by checking the current date and time.
Tool Calls:
  current_datetime (call_00_IrXgeKKSQ0W153SLJzrL4600)
 Call ID: call_00_IrXgeKKSQ0W153SLJzrL4600
  Args:
================================= Tool Message =====